# Notebook 7: Cross-Scenario Evaluation

Run all fusion attacks across all available scenarios and compare results.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path
from data_loader import SensorDataLoader
from attacks.fusion_attacks import FusionAttacker, FusionAttackType
from defenses.defense_mechanisms import DefensePipeline, DefenseType, AdversarialTraining
from evaluation.metrics import TrackingMetrics

%matplotlib inline

## 7.1 Discover Available Scenarios

In [ ]:
data_path = Path('../data/sensor_fusion_dataset')
scenarios = sorted([d.name for d in data_path.iterdir() if d.is_dir() and 'scenario' in d.name])
print(f"Found {len(scenarios)} scenarios: {scenarios}")

## 7.2 Run All Fusion Attacks on All Scenarios

In [ ]:
attacker = FusionAttacker()
metrics = TrackingMetrics()
pipeline = DefensePipeline()

all_results = []

for scenario in scenarios[:3]:  # Limit to first 3 for demo
    print(f"\n=== {scenario} ===")
    path = f'../data/sensor_fusion_dataset/{scenario}'
    
    try:
        loader = SensorDataLoader(path)
        detections = loader.load_all_detections()
        ground_truth = loader.load_ground_truth()
        
        for attack_type in FusionAttackType:
            # Attack
            attacked = attacker.attack_scenario(detections, attack_type, ground_truth)
            
            # Defend
            defended = pipeline.defend_detections(attacked, DefenseType.TEMPORAL_CONSISTENCY, ground_truth)
            
            # Metrics for IR camera
            m_benign = metrics.compute_sensor_metrics(detections[3], ground_truth, 3)
            m_attacked = metrics.compute_sensor_metrics(attacked[3], ground_truth, 3)
            m_defended = metrics.compute_sensor_metrics(defended[3], ground_truth, 3)
            
            all_results.append({
                'scenario': scenario,
                'attack': attack_type.name,
                'benign_detprob': m_benign['detection_probability'],
                'attacked_detprob': m_attacked['detection_probability'],
                'defended_detprob': m_defended['detection_probability'],
                'benign_far': m_benign['false_alarm_rate'],
                'attacked_far': m_attacked['false_alarm_rate'],
                'defended_far': m_defended['false_alarm_rate'],
            })
        
        print(f"  Processed {len(FusionAttackType)} attacks")
        
    except Exception as e:
        print(f"  Error: {e}")

df_results = pd.DataFrame(all_results)
print(f"\nTotal results: {len(df_results)} rows")

## 7.3 Attack Effectiveness Heatmap

In [ ]:
# Create heatmap of detection probability after attack
pivot = df_results.pivot_table(
    values='attacked_detprob', 
    index='attack', 
    columns='scenario',
    aggfunc='mean'
)

plt.figure(figsize=(10, 8))
plt.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
plt.colorbar(label='Detection Probability After Attack')
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45)
plt.yticks(range(len(pivot.index)), [a.replace('_', ' ') for a in pivot.index])
plt.title('Attack Effectiveness Heatmap\n(Lower = More Effective Attack)')
plt.tight_layout()
plt.show()

## 7.4 Defense Recovery Heatmap

In [ ]:
# Compute recovery rate
df_results['recovery'] = ((df_results['defended_detprob'] - df_results['attacked_detprob']) / 
                          (df_results['benign_detprob'] - df_results['attacked_detprob']) * 100)
df_results['recovery'] = df_results['recovery'].clip(0, 100)

pivot_recovery = df_results.pivot_table(
    values='recovery', 
    index='attack', 
    columns='scenario',
    aggfunc='mean'
)

plt.figure(figsize=(10, 8))
plt.imshow(pivot_recovery.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)
plt.colorbar(label='Defense Recovery (%)')
plt.xticks(range(len(pivot_recovery.columns)), pivot_recovery.columns, rotation=45)
plt.yticks(range(len(pivot_recovery.index)), [a.replace('_', ' ') for a in pivot_recovery.index])
plt.title('Defense Recovery Heatmap\n(Higher = Better Defense)')
plt.tight_layout()
plt.show()

## 7.5 Scenario Comparison Summary

In [ ]:
# Aggregate by scenario
scenario_summary = df_results.groupby('scenario').agg({
    'benign_detprob': 'mean',
    'attacked_detprob': 'mean',
    'defended_detprob': 'mean',
    'recovery': 'mean'
}).reset_index()

print(scenario_summary.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(scenario_summary))
width = 0.2

ax.bar(x - width, scenario_summary['benign_detprob'] * 100, width, label='Benign', color='blue', alpha=0.7)
ax.bar(x, scenario_summary['attacked_detprob'] * 100, width, label='Attacked', color='red', alpha=0.7)
ax.bar(x + width, scenario_summary['defended_detprob'] * 100, width, label='Defended', color='green', alpha=0.7)

ax.set_xlabel('Scenario')
ax.set_ylabel('Detection Probability (%)')
ax.set_title('Average Performance Across All Fusion Attacks')
ax.set_xticks(x)
ax.set_xticklabels(scenario_summary['scenario'])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7.6 Save Results

In [ ]:
# Save to JSON
output_file = '../results/cross_scenario_analysis.json'
df_results.to_json(output_file, orient='records', indent=2)
print(f"Results saved to {output_file}")

# Also save summary
summary_file = '../results/cross_scenario_summary.json'
scenario_summary.to_json(summary_file, orient='records', indent=2)
print(f"Summary saved to {summary_file}")